In [ ]:
import sys

import matplotlib.pyplot as plt
import pandas as pd
import copy

sys.path.append('/uufs/chpc.utah.edu/common/home/u6058223/git_dirs/env/')
import helpers as h

import seaborn as sns
sns.set_palette('icefire')

import xarray as xr

In [ ]:
aso_dir = '/uufs/chpc.utah.edu/common/home/skiles-group3/ASO/UT'
aso_fns = h.fn_list(aso_dir, '*tif')
aso_fns

In [ ]:
# Read them in and plot them
ds_list = [h.load(fn) for fn in aso_fns]

In [ ]:
def process_snotel_por_bynums(site_nums, rows2skip=65):
    # Pull in all the csvs in this order that match at least that site
    snotel_dir = '/uufs/chpc.utah.edu/common/home/skiles-group3/ancillary_sdswe_products/SNOTEL'
    snotel_fns = []
    for site in site_nums:
        site_fns = h.fn_list(snotel_dir, f'*{site}*.csv')
        # this way, no need to index with fn_list, can just extend the entire list
        snotel_fns.extend(site_fns)
    # Read them into a big snotel_df list
    snotel_dfs = [pd.read_csv(fn, skiprows=rows2skip) for fn in snotel_fns]

    # Drop all columns from the df except for the date and snow depth
    for i, df in enumerate(snotel_dfs):
        df = df.iloc[:, [0, 1]]
        print(df.columns)
        # Set the date column as the index
        df.set_index(df.columns[0], inplace=True)
        snotel_dfs[i] = df

    # Note this is all in centimeters! needs converting for the plotting
    # Concatenate all the snotel_dfs into one big df using the date as the index
    snotel_df = pd.concat(snotel_dfs, axis=1)
    # convert the date index to datetime
    snotel_df.index = pd.to_datetime(snotel_df.index)
    # Arrange the index such that it is chronological
    snotel_df.sort_index(inplace=True)
    return snotel_df

In [ ]:
# Plot the SNOTEL site data in the overlapping area
# There's code for this, but I'm just going to eyeball for expedience right now
# site_nums = [814, 1225, 763, 392, 393]
# For the Lost Creek one
site_nums = [1118]
snotel_df = process_snotel_por_bynums(site_nums)
snotel_df

In [ ]:
# Trim to 2025 and 2026
snotel_df = snotel_df[(snotel_df.index >= '2024-10-01') & (snotel_df.index <= '2026-04-21')]
snotel_df

In [ ]:
snotel_df.plot(figsize=(15, 6), linestyle='--', lw=1)

In [ ]:
# Plot a single SNOTEL site
snotel_df.iloc[:, 0].plot(figsize=(15, 6), linestyle='--', lw=1)
# add snotel site name to the plot
plt.title(snotel_df.columns[0])

In [ ]:
# Pull the metadata from the active site table in the snotel dir
snotel_dir = '/uufs/chpc.utah.edu/common/home/skiles-group3/ancillary_sdswe_products/SNOTEL'
site_table_df = pd.read_csv(h.fn_list(snotel_dir, '*active*csv')[0], index_col=0)
# Subset by the site numbers in site_nums
site_table_df = site_table_df[site_table_df['site_num'].isin(site_nums)]
# convert elevation to meters in a new column
site_table_df['elev_m'] = site_table_df['elev'] * 0.3048
site_table_df

In [ ]:
# Pull in the topo.nc for the Weber to grab a bit of elevation data
topo_fn = '/uufs/chpc.utah.edu/common/home/skiles-group3/jmhu/isnobal_scripts/weber_scripts/topo.nc'
topo_ds = xr.open_dataset(topo_fn)
topo_ds

In [ ]:
# Plot the ASO data at the SNOTEL 


In [ ]:
# Sample the ASO data at the SNOTEL sites and plot them on top of the SNOTEL data by subplots